In [ ]:
# Importation of libraries and setting of seaborn theme
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

sns.set_theme(context='talk', style='whitegrid', palette='deep', font='sans-serif')

In [ ]:
# Reading of both the pre-processing datasets
df_tr = pd.read_csv("LB2_project_Group_7/0.2_data_pre_processing/0.2.2_train/training_annotated.tsv", sep='\t')
df_bn = pd.read_csv("drive/MyDrive/Colab Notebooks/benchmarking_annotated.tsv", sep='\t')

In [ ]:
# PROTEIN LENGTHS DISTRIBUTION - TRAINING
df = df_tr.dropna(subset=["label"])
df["label"] = df["label"].astype(str)     # We set labels as strings

g = sns.displot(
    data=df,
    x="length",
    hue="label",                                      # This way, we separate positives and negatives
    log_scale=True,
    bins=40,
    height=6,
    aspect=1.5
)
g.set_axis_labels("Sequence length (aa)", "Frequency")
g.fig.suptitle("Distribution of protein lengths (training)\n", fontsize = 12)
plt.show()

In [ ]:
# PROTEIN LENGTHS DISTRIBUTION - BENCHMARKING
df = df_bn.dropna(subset=["label"])
df["label"] = df["label"].astype(str)     # We set labels as strings

g = sns.displot(
    data=df,
    x="length",
    hue="label",                                      # This way, we separate positives and negatives
    log_scale=True,
    bins=40,
    height=6,
    aspect=1.5
)
g.set_axis_labels("Sequence length (aa)", "Frequency")
g.fig.suptitle("Distribution of protein lengths (benchmarking)", fontsize = 12)
plt.show()

In [ ]:
# SP LENGTHS DISTRIBUTION - TRAINING
df = df_tr.dropna(subset=["label"])
df["label"] = df["label"].astype(int).astype(str)       # We set labels as strings
positives = df[df['label'] == '1']

g = sns.displot(
    data=positives,
    x="cleavage_site",
    hue="label",                                        # This way, we separate positives and negatives
    bins=40,
    height=6,
    aspect=1.5,
)
g.set_axis_labels("SP length (aa)", "Frequency")
g.fig.suptitle("Distribution of SP lengths (training)", fontsize = 12)
plt.show()

In [ ]:
# SP LENGTHS DISTRIBUTION - BENCHMARKING
df = df_bn.dropna(subset=["label"])
df["label"] = df["label"].astype(int).astype(str)       # We set labels as strings
positives = df[df['label'] == '1']

g = sns.displot(
    data=positives,
    x="cleavage_site",
    hue="label",                                        # This way, we separate positives and negatives
    bins=40,
    height=6,
    aspect=1.5,
)
g.set_axis_labels("SP length (aa)", "Frequency")
g.fig.suptitle("Distribution of SP lengths (benchmarking)", fontsize = 12)
plt.show()

In [ ]:
# COMPARATIVE ANALYSIS OF SP AMINOACID COMPOSITION - TRAINING
background = {
    "L": 9.65, "A": 8.26, "G": 7.07, "V": 6.85, "E": 6.71,
    "S": 6.67, "I": 5.90, "K": 5.79, "R": 5.53, "D": 5.46,
    "T": 5.37, "P": 4.75, "N": 4.06, "Q": 3.93, "F": 3.87,
    "Y": 2.92, "M": 2.41, "H": 2.28, "C": 1.39, "W": 1.11,
}

count = {aa: 0 for aa in background.keys()}

df = df_tr.dropna(subset=["sequence"])
positives = df[df["label"] == 1]

# Simultaneously checks sequence and cleavage site position to include only the SP
for seq, cleavage in zip(positives["sequence"], positives["cleavage_site"]):
    sp = seq[:int(cleavage)]
    for aa in sp:
        if aa in count:
            count[aa] += 1

# Converts counts to percentage
total = sum(count.values())
sp_percent = {aa: 100 * n / total for aa, n in count.items()}

comp = pd.DataFrame({
    "aa": list(background),
    "SP": [sp_percent[aa] for aa in background],
    "Background": list(background.values()),
})
comp_long = comp.melt(id_vars="aa", var_name="Origin", value_name="pct")

plt.figure(figsize=(14, 6))
sns.barplot(data=comp_long, x="aa", y="pct", hue="Origin")
plt.xlabel("Aminoacid")
plt.ylabel("Composition (%)")
plt.title("Comparative aminoacid composition of SPs (training)")
plt.show()


In [ ]:
# COMPARATIVE ANALYSIS OF SP AMINOACID COMPOSITION - BENCHMARKING
background = {
    "L": 9.65, "A": 8.26, "G": 7.07, "V": 6.85, "E": 6.71,
    "S": 6.67, "I": 5.90, "K": 5.79, "R": 5.53, "D": 5.46,
    "T": 5.37, "P": 4.75, "N": 4.06, "Q": 3.93, "F": 3.87,
    "Y": 2.92, "M": 2.41, "H": 2.28, "C": 1.39, "W": 1.11,
}

count = {aa: 0 for aa in background.keys()}

df = df_bn.dropna(subset=["sequence"])
positives = df[df["label"] == 1]

# Simultaneously checks sequence and cleavage site position to include only the SP
for seq, cleavage in zip(positives["sequence"], positives["cleavage_site"]):
    sp = seq[:int(cleavage)]
    for aa in sp:
        if aa in count:
            count[aa] += 1

# Converts counts to percentage
total = sum(count.values())
sp_percent = {aa: 100 * n / total for aa, n in count.items()}

comp = pd.DataFrame({
    "aa": list(background),
    "SP": [sp_percent[aa] for aa in background],
    "Background": list(background.values()),
})
comp_long = comp.melt(id_vars="aa", var_name="Origin", value_name="pct")

plt.figure(figsize=(14, 6))
sns.barplot(data=comp_long, x="aa", y="pct", hue="Origin")
plt.xlabel("Aminoacid")
plt.ylabel("Composition (%)")
plt.title("Comparative aminoacid composition of SPs (benchmarking)")
plt.show()

In [ ]:
# TAXONOMIC CLASSIFICATION (Kingdoms and most frequent species) - TRAINING
df = df_tr.dropna(subset=["organism"])

kingdom_colors = {
    "Metazoa": "#4C72B0",
    "Plants":  "#55A868",
    "Fungi":   "#C44E52",
    "Other":   "#8172B2",
}

# Top-N approach to show the most frequent species per kingdom, the rest are grouped as "Other"
TOP_N_SPECIES = 8

def plot_kingdom_pie(data, title):
    """Pie chart of the kingdom distribution."""
    counts = data["kingdom"].value_counts()
    colors = [kingdom_colors[k] for k in counts.index]
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=90, colors = colors)
    ax.set_title(title)
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

def plot_species_by_kingdom(data, title, top_n=TOP_N_SPECIES):
    """One bar panel per kingdom, with the top N species and an 'Other' bar for the rest."""
    kingdoms = data["kingdom"].value_counts().index
    n = len(kingdoms)
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 6))
    if n == 1:
        axes = [axes]
    for ax, kingdom in zip(axes, kingdoms):
        sub = data[data["kingdom"] == kingdom]
        scientific_name = sub["organism"].str.split().str[:2].str.join(" ")
        counts = scientific_name.value_counts()
        if len(counts) > top_n:
            top = counts.iloc[:top_n]
            other = pd.Series({"Other": counts.iloc[top_n:].sum()})
            counts = pd.concat([top, other])
        sns.barplot(x=counts.values, y=counts.index, ax=ax, color="C0")
        ax.set_title(kingdom)
        ax.set_xlabel("Number of sequences")
        ax.set_ylabel("")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

for label, name in [(1, "Positives"), (0, "Negatives")]:
    subset = df[df["label"] == label]
    plot_kingdom_pie(subset, f"Kingdom distribution — {name} (training)")
    plot_species_by_kingdom(subset, f"Top species by kingdom — {name} (training)")

In [ ]:
# TAXONOMIC CLASSIFICATION (Kingdoms and most frequent species) - BENCHMARKING
df = df_bn.dropna(subset=["organism"])

for label, name in [(1, "Positives"), (0, "Negatives")]:
    subset = df[df["label"] == label]
    plot_kingdom_pie(subset, f"Kingdom distribution — {name} (benchmarking)")
    plot_species_by_kingdom(subset, f"Top species by kingdom — {name} (benchmarking)")